# 09-10 Colab Sparse Autoencoder And Text-To-Brain Pipeline

Run-all Colab notebook for Stage 1 sparse autoencoder and Stage 2 text-to-brain projection variants. It clones the repo and copies generated data from Drive automatically.


## 1. Runtime And Drive Mount

This notebook is designed for Colab Run all. It clones the repository into `/content/neurovlm`, copies your generated data from Google Drive into the clone, and writes run outputs back to Drive.


In [ ]:
import os, sys, json, time, platform, subprocess, shutil
from pathlib import Path

print('Python:', sys.version)
print('Platform:', platform.platform())
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout if gpu.returncode == 0 else 'No GPU detected by nvidia-smi. Use Runtime -> Change runtime type -> GPU.')

from google.colab import drive
drive.mount('/content/drive')


## 2. Clone Or Pull Repository

In [ ]:
REPO_URL = 'https://github.com/neurovlm/neurovlm.git'
REPO_BRANCH = 'neurovlm_gnn'
REPO_DIR = Path('/content/neurovlm')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=False)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repo ready at', REPO_DIR)
print(subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip())


## 3. Install Dependencies

In [ ]:
%pip install -q -e /content/neurovlm
%pip install -q torch pandas numpy nibabel nilearn pyyaml tqdm safetensors transformers adapters pyarrow matplotlib scikit-learn


## 4. Copy Generated Data From Drive Into The Clone

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive')
PREFERRED_DRIVE_PROJECT_DIR = DRIVE_ROOT / 'neurovlm'

required_rel = [
    'atlas_free_multipositive/cache/unified_jsonl/splits/train.jsonl',
    'atlas_free_multipositive/cache/unified_jsonl/splits/val.jsonl',
    'atlas_free_multipositive/cache/unified_jsonl/text_registry.jsonl',
    'atlas_free_multipositive/cache/text_embeddings/specter_text_cache.pt',
    'atlas_free_multipositive/data/ale_caches/atlas_free_4mm_fwhm9_crop_float16.pt',
]

def has_generated_data(root: Path) -> bool:
    return all((root / rel).exists() for rel in required_rel)

candidates = []
if has_generated_data(PREFERRED_DRIVE_PROJECT_DIR):
    candidates.append(PREFERRED_DRIVE_PROJECT_DIR)
for train_json in DRIVE_ROOT.glob('**/atlas_free_multipositive/cache/unified_jsonl/splits/train.jsonl'):
    root = train_json.parents[4]
    if has_generated_data(root) and root not in candidates:
        candidates.append(root)

if not candidates:
    print('Searched for generated data under', DRIVE_ROOT)
    raise FileNotFoundError('Could not find generated atlas_free_multipositive data on Drive. Put it under /content/drive/MyDrive/neurovlm or another folder with the same relative paths.')

DRIVE_PROJECT_DIR = candidates[0]
print('Using generated data from', DRIVE_PROJECT_DIR)

copy_roots = [
    'atlas_free_multipositive/cache',
    'atlas_free_multipositive/data/ale_caches',
]
for rel in copy_roots:
    src = DRIVE_PROJECT_DIR / rel
    dst = REPO_DIR / rel
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print('copied', src, '->', dst)

DRIVE_RUNS_DIR = DRIVE_PROJECT_DIR / 'atlas_free_multipositive/outputs/runs'
DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)
print('Drive run outputs:', DRIVE_RUNS_DIR)


## 5. Verify Local Training Files

In [ ]:
missing = []
for rel in required_rel:
    p = REPO_DIR / rel
    ok = p.exists()
    print(f'{str(ok):5}  {p}')
    if not ok:
        missing.append(str(p))
if missing:
    raise FileNotFoundError('Missing local files after Drive copy: ' + repr(missing))


## 6. Imports And Stage Settings

In [ ]:
import copy, yaml, torch
from atlas_free_multipositive.training.train_autoencoder_sparse import train_from_config as train_sparse_autoencoder
from atlas_free_multipositive.training.train_text_to_brain import train_from_config as train_text_to_brain
BASE_RUN_DIR = DRIVE_RUNS_DIR / time.strftime('colab_generation_%Y%m%d_%H%M%S')
BASE_RUN_DIR.mkdir(parents=True, exist_ok=True)
print('BASE_RUN_DIR =', BASE_RUN_DIR)


## 7. Stage 1 - Sparse CNN Autoencoder

In [ ]:
ae_cfg = yaml.safe_load(Path('atlas_free_multipositive/configs/autoencoder_sparse_config.yaml').read_text())
ae_cfg.update({'output_dir': str(BASE_RUN_DIR / 'stage1_sparse_autoencoder'), 'checkpoint_dir': str(BASE_RUN_DIR / 'stage1_sparse_autoencoder' / 'checkpoints'), 'device': 'auto', 'epochs': 5, 'batch_size': 8, 'max_val_batches': 50})
ae_result = train_sparse_autoencoder(ae_cfg)
AE_CHECKPOINT = Path(ae_cfg['checkpoint_dir']) / 'best_generation_top5_dice.pt'
if not AE_CHECKPOINT.exists():
    AE_CHECKPOINT = Path(ae_cfg['checkpoint_dir']) / 'last.pt'
print('AE_CHECKPOINT =', AE_CHECKPOINT)


## 8. Stage 2A - Text-To-Brain Projection, Random Init

In [ ]:
base_t2b_cfg = yaml.safe_load(Path('atlas_free_multipositive/configs/text_to_brain_config.yaml').read_text())
t2b_random = copy.deepcopy(base_t2b_cfg)
t2b_random.update({'autoencoder_checkpoint': str(AE_CHECKPOINT), 'text_projection_init': 'random', 'output_dir': str(BASE_RUN_DIR / 'stage2_text_to_brain_random'), 'checkpoint_dir': str(BASE_RUN_DIR / 'stage2_text_to_brain_random' / 'checkpoints'), 'device': 'auto', 'epochs': 5, 'batch_size': 4, 'max_val_batches': 50})
random_result = train_text_to_brain(t2b_random)
RANDOM_T2B_CHECKPOINT = Path(t2b_random['checkpoint_dir']) / 'best_generation_top5_dice.pt'
if not RANDOM_T2B_CHECKPOINT.exists():
    RANDOM_T2B_CHECKPOINT = Path(t2b_random['checkpoint_dir']) / 'last.pt'
print('RANDOM_T2B_CHECKPOINT =', RANDOM_T2B_CHECKPOINT)


## 9. Stage 2B - Text-To-Brain Projection, Pretrained Text InfoNCE Init

In [ ]:
t2b_pretrained = copy.deepcopy(base_t2b_cfg)
t2b_pretrained.update({'autoencoder_checkpoint': str(AE_CHECKPOINT), 'text_projection_init': 'pretrained_text_infonce', 'output_dir': str(BASE_RUN_DIR / 'stage2_text_to_brain_pretrained_text_infonce'), 'checkpoint_dir': str(BASE_RUN_DIR / 'stage2_text_to_brain_pretrained_text_infonce' / 'checkpoints'), 'device': 'auto', 'epochs': 5, 'batch_size': 4, 'max_val_batches': 50})
pretrained_result = train_text_to_brain(t2b_pretrained)
PRETRAINED_T2B_CHECKPOINT = Path(t2b_pretrained['checkpoint_dir']) / 'best_generation_top5_dice.pt'
if not PRETRAINED_T2B_CHECKPOINT.exists():
    PRETRAINED_T2B_CHECKPOINT = Path(t2b_pretrained['checkpoint_dir']) / 'last.pt'
print('PRETRAINED_T2B_CHECKPOINT =', PRETRAINED_T2B_CHECKPOINT)


## 10. Save Manifest

In [ ]:
manifest = {'base_run_dir': str(BASE_RUN_DIR), 'autoencoder_checkpoint': str(AE_CHECKPOINT), 'random_text_to_brain_checkpoint': str(RANDOM_T2B_CHECKPOINT), 'pretrained_text_infonce_text_to_brain_checkpoint': str(PRETRAINED_T2B_CHECKPOINT), 'stage1_result': ae_result, 'stage2_random_result': random_result, 'stage2_pretrained_result': pretrained_result}
json.dump(manifest, open(BASE_RUN_DIR / 'generation_stage_manifest.json', 'w'), indent=2)
print(json.dumps(manifest, indent=2))
